# TRM Maze - 10×10 Image Dataset (OPTIMIZED)

**Dataset:** Kaggle rectangular mazes 10×10  
**Approach:** Direction prediction (vocab=5)  
**Target:** Train in 2-3 hours, 50-60% accuracy  

**Key Optimizations:**
- ✓ Small 10×10 grid (vocab=100 for positions, 5 for directions)
- ✓ Direct image → array conversion (no vision encoder needed)
- ✓ Direction prediction (UP/DOWN/LEFT/RIGHT/STOP)
- ✓ Smaller model (faster training)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install torch numpy pillow kaggle opencv-python -q

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
import numpy as np, time, os, glob, cv2
from collections import deque
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Model (Compact Version)

In [ ]:
# Lightweight model components
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))
    def forward(self, x): return x / torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        h = int(8/3 * dim)  # Smaller hidden dim
        self.w1, self.w2, self.w3 = nn.Linear(dim,h,bias=False), nn.Linear(h,dim,bias=False), nn.Linear(dim,h,bias=False)
    def forward(self, x): return self.w2(F.silu(self.w1(x)) * self.w3(x))

class TinyAttention(nn.Module):
    def __init__(self, dim, n_heads=4):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, dim//n_heads
        self.scale = self.head_dim**-0.5
        self.qkv = nn.Linear(dim, 3*dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv(x).reshape(B,L,3,self.n_heads,self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = F.softmax((q @ k.transpose(-2,-1))*self.scale, dim=-1)
        return self.out((attn @ v).transpose(1,2).reshape(B,L,D))

class TinyBlock(nn.Module):
    def __init__(self, dim, n_heads=4):
        super().__init__()
        self.norm1, self.attn = RMSNorm(dim), TinyAttention(dim, n_heads)
        self.norm2, self.ffn = RMSNorm(dim), SwiGLU(dim)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class TinyNet(nn.Module):
    def __init__(self, dim=256, n_layers=2, n_heads=4):
        super().__init__()
        self.layers = nn.ModuleList([TinyBlock(dim, n_heads) for _ in range(n_layers)])
        self.norm = RMSNorm(dim)
    def forward(self, x):
        for layer in self.layers: x = layer(x)
        return self.norm(x)
    def count_parameters(self): return sum(p.numel() for p in self.parameters())

class LatentRecursion(nn.Module):
    def __init__(self, net, dim=256, n_latent=4):
        super().__init__()
        self.net, self.n_latent = net, n_latent
        self.z_proj = nn.Linear(3*dim, dim, bias=False)
        self.y_proj = nn.Linear(2*dim, dim, bias=False)
    def forward(self, x, y, z):
        for _ in range(self.n_latent): z = self.net(self.z_proj(torch.cat([x,y,z],dim=-1)))
        return self.net(self.y_proj(torch.cat([y,z],dim=-1))), z

class TRMModel(nn.Module):
    def __init__(self, dim=256, n_layers=2, n_heads=4, n_latent=4, T_cycles=2, vocab_size=5):
        super().__init__()
        self.T_cycles = T_cycles
        self.net = TinyNet(dim, n_layers, n_heads)
        self.latent_recursion = LatentRecursion(self.net, dim, n_latent)
        self.output_head = nn.Linear(dim, vocab_size, bias=False)
    def forward(self, x, y, z):
        with torch.no_grad():
            for _ in range(self.T_cycles-1): y, z = self.latent_recursion(x, y, z)
        y, z = self.latent_recursion(x, y, z)
        return (y,z), self.output_head(y)
    def count_parameters(self): return sum(p.numel() for p in self.parameters())

## Load 10×10 Image Mazes

In [ ]:
# Download dataset
os.environ['KAGGLE_USERNAME'] = 'aayushbhure'
os.environ['KAGGLE_KEY'] = 'YOUR_KEY_HERE'

!kaggle datasets download -d emadehsan/rectangular-maze-kruskals-spanning-tree-algorithm
!unzip -q rectangular-maze-kruskals-spanning-tree-algorithm.zip

print("✓ Dataset downloaded")

In [ ]:
# Image to maze array converter
def image_to_maze(img_path):
    """Convert maze image to binary array (0=path, 1=wall)"""
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    # Threshold: white (255) = path (0), black (0) = wall (1)
    _, binary = cv2.threshold(img, 127, 1, cv2.THRESH_BINARY_INV)
    return binary

def path_to_directions(path):
    """Convert path to directions: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT, 4=STOP"""
    if len(path) < 2: return [4]
    dirs = []
    for i in range(len(path)-1):
        r1, c1, r2, c2 = path[i][0], path[i][1], path[i+1][0], path[i+1][1]
        if r2 > r1: dirs.append(1)  # DOWN
        elif r2 < r1: dirs.append(0)  # UP
        elif c2 > c1: dirs.append(3)  # RIGHT
        elif c2 < c1: dirs.append(2)  # LEFT
    dirs.append(4)  # STOP
    return dirs

def find_path_bfs(maze):
    rows, cols = maze.shape
    start, goal = (0,0), (rows-1, cols-1)
    queue, visited = deque([(start, [start])]), {start}
    while queue:
        (r,c), path = queue.popleft()
        if (r,c) == goal: return path
        for dr,dc in [(0,1),(1,0),(0,-1),(-1,0)]:
            nr, nc = r+dr, c+dc
            if 0<=nr<rows and 0<=nc<cols and maze[nr,nc]==0 and (nr,nc) not in visited:
                visited.add((nr,nc))
                queue.append(((nr,nc), path+[(nr,nc)]))
    return [start, goal]

# Load all 10×10 mazes
print("Loading 10×10 mazes...")
train_data = []
image_files = glob.glob('rectangular_mazes_10x10/*.png')[:3000]

for img_path in image_files:
    try:
        maze = image_to_maze(img_path)
        if maze.shape != (10, 10): continue
        
        path = find_path_bfs(maze)
        if len(path) < 2: continue
        
        directions = path_to_directions(path[:30])  # Max 30 steps
        
        train_data.append({
            "maze": maze.flatten().tolist(),
            "directions": directions
        })
        
        if len(train_data) % 500 == 0:
            print(f"  Loaded {len(train_data)} mazes...")
    except:
        continue

print(f"\n✓ Loaded {len(train_data)} valid 10×10 mazes")
print(f"  Avg path length: {np.mean([len(d['directions']) for d in train_data]):.1f} steps")
print(f"  Sample directions: {train_data[0]['directions'][:10]}")

In [ ]:
class MazeDataset(Dataset):
    def __init__(self, data, max_len=128):
        self.data, self.max_len = data, max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        x = torch.zeros(self.max_len, dtype=torch.long)
        x[:100] = torch.tensor(item["maze"], dtype=torch.long)  # 10×10=100
        y_init = torch.zeros(self.max_len, dtype=torch.long)
        y_true = torch.zeros(self.max_len, dtype=torch.long)
        dirs = item["directions"][:30]
        y_true[:len(dirs)] = torch.tensor(dirs, dtype=torch.long)
        return {"x_tokens": x, "y_init_tokens": y_init, "y_true": y_true, "path_len": len(dirs)}

## Training Setup (FAST)

In [ ]:
# OPTIMIZED CONFIG
CONFIG = {
    "dim": 256,        # Smaller model
    "n_layers": 2,
    "n_heads": 4,
    "n_latent": 4,     # Reduced
    "T_cycles": 2,     # Reduced
    "vocab_size": 5,
    "max_seq_len": 128,
    "lr": 1e-3,        # Higher LR
    "weight_decay": 0.01,
    "max_iters": 6000,
    "batch_size": 64,  # Larger batches
    "warmup_steps": 300,
    "grad_clip": 1.0,
    "N_sup": 3,        # Reduced
    "ema_decay": 0.999,
    "save_every": 1000,
    "log_every": 50
}

dataset = MazeDataset(train_data)
dataloader = DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2)

model = TRMModel(CONFIG["dim"], CONFIG["n_layers"], CONFIG["n_heads"], 
                 CONFIG["n_latent"], CONFIG["T_cycles"], CONFIG["vocab_size"]).to(device)
embedding = nn.Embedding(10, CONFIG["dim"]).to(device)
optimizer = optim.AdamW(list(model.parameters())+list(embedding.parameters()), 
                        lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = LambdaLR(optimizer, lambda s: min(1.0, s/CONFIG["warmup_steps"]))
scaler = torch.amp.GradScaler('cuda')

class EMA:
    def __init__(self, model, decay=0.999):
        self.model, self.decay, self.shadow, self.backup = model, decay, {}, {}
        for n,p in model.named_parameters():
            if p.requires_grad: self.shadow[n] = p.data.clone()
    def update(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad: self.shadow[n] = self.decay*self.shadow[n] + (1-self.decay)*p.data
    def apply_shadow(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad: self.backup[n], p.data = p.data.clone(), self.shadow[n]
    def restore(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad: p.data = self.backup[n]

ema_model, ema_embedding = EMA(model), EMA(embedding)

print(f"Model parameters: {model.count_parameters():,}")
print(f"Expected time: ~2 hours")
print(f"Target accuracy: 50-60%")

## Training Loop (WITH PATH LENGTH ACCURACY)

In [ ]:
os.makedirs('/content/drive/MyDrive/TRM_Maze_10x10', exist_ok=True)
print("Starting 10×10 maze training...\n" + "="*60)

model.train()
train_iter, start_time, best_acc = iter(dataloader), time.time(), 0.0

for step in range(CONFIG["max_iters"]):
    try: batch = next(train_iter)
    except StopIteration: train_iter, batch = iter(dataloader), next(iter(dataloader))
    
    x_tokens = batch["x_tokens"].to(device)
    y_true = batch["y_true"].to(device)
    path_lens = batch["path_len"]
    y_tokens = batch["y_init_tokens"].to(device)
    
    for _ in range(CONFIG["N_sup"]):
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            x, y, z = embedding(x_tokens), embedding(y_tokens), torch.zeros(x_tokens.size(0), CONFIG["max_seq_len"], CONFIG["dim"], device=device)
            (_, _), y_hat = model(x, y, z)
            loss = F.cross_entropy(y_hat.view(-1, y_hat.size(-1)), y_true.view(-1), ignore_index=0)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(list(model.parameters())+list(embedding.parameters()), CONFIG["grad_clip"])
        scaler.step(optimizer); scaler.update(); scheduler.step()
        ema_model.update(); ema_embedding.update()
        with torch.no_grad(): y_tokens = y_hat.argmax(dim=-1)
    
    if step % CONFIG["log_every"] == 0:
        with torch.no_grad():
            ema_model.apply_shadow(); ema_embedding.apply_shadow()
            x, y, z = embedding(x_tokens), embedding(batch["y_init_tokens"].to(device)), torch.zeros_like(x)
            (_, _), y_hat = model(x, y, z)
            
            # CORRECTED ACCURACY: Only count actual path tokens
            preds = y_hat.argmax(dim=-1)
            correct = 0
            total = 0
            for i in range(len(path_lens)):
                plen = path_lens[i]
                correct += (preds[i, :plen] == y_true[i, :plen]).sum().item()
                total += plen
            acc = correct / max(total, 1)
            
            ema_model.restore(); ema_embedding.restore()
        
        best_acc = max(best_acc, acc)
        print(f"Step {step:4d} | Loss: {loss.item():.4f} | Acc: {acc:.3f} | Best: {best_acc:.3f} | Time: {time.time()-start_time:.0f}s")
    
    if step>0 and step%CONFIG["save_every"]==0:
        ema_model.apply_shadow(); ema_embedding.apply_shadow()
        torch.save({"model": model.state_dict(), "embedding": embedding.state_dict(), 
                    "config": CONFIG, "best_acc": best_acc},
                   f"/content/drive/MyDrive/TRM_Maze_10x10/step_{step}.pt")
        ema_model.restore(); ema_embedding.restore()
        print(f"  ✓ Saved checkpoint")

# Final save
ema_model.apply_shadow(); ema_embedding.apply_shadow()
torch.save({"model": model.state_dict(), "embedding": embedding.state_dict(), 
            "config": CONFIG, "best_acc": best_acc}, 
           "/content/drive/MyDrive/TRM_Maze_10x10/final.pt")
print(f"\nComplete! Best accuracy: {best_acc:.1%}")
print(f"Target was 50-60%, {'✓ ACHIEVED!' if best_acc >= 0.5 else 'Continue training' if best_acc > 0.3 else 'Check implementation'}")